# SmartX demo — zero-shot classification & image–text retrieval

**SmartX** is an open vision–language foundation model for real-world multi-anatomy X-ray interpretation.
It pairs a Swin Transformer image encoder with a PubMedBERT text encoder in a shared embedding space. This demo uses chest X-rays.

This notebook shows the two things SmartX can do **using the pretrained weights alone — no
fine-tuning, no training loop**:

1. **Zero-shot classification** — score an image against text prompts for each finding.
2. **Image–text retrieval** — rank radiology reports for an image (and vice versa) by
   embedding similarity.

All heavy lifting goes through the shared loader `create_clip_style_model_global`, the same
API used by `downstream/zero_shot_classification` and `downstream/retrieval`. The demo images are
public-domain chest X-rays (see `assets/README.md`); the reports are short synthetic
radiology-style captions written for the demo.

> **Requirements:** the SmartX checkpoint at `checkpoints/smartx_large.ckpt`
> (see `checkpoints/README.md` / `docs/model_zoo.md`). Runs on CPU or GPU.

## 1. Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Make the repo importable when running from demo/ or the repo root.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "downstream").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

ASSETS = REPO_ROOT / "demo" / "assets"

# Use SMARTX_CHECKPOINT if set, otherwise a local checkpoints/smartx_large.ckpt if present,
# or None to auto-download from Hugging Face `Trusure/SmartX`.
_default_ckpt = REPO_ROOT / "checkpoints" / "smartx_large.ckpt"
CHECKPOINT = os.environ.get("SMARTX_CHECKPOINT") or (str(_default_ckpt) if _default_ckpt.is_file() else None)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 768  # SmartX's Swin encoder is built for 768x768 inputs.
print("repo root:", REPO_ROOT)
print("device:", DEVICE)
print("checkpoint:", CHECKPOINT)

## 2. Load the model

`create_clip_style_model_global` returns L2-normalized image and text encoders plus the
tokenizer. `model_info` reports the embedding dimensions and how cleanly the checkpoint
loaded.

In [ ]:
from downstream.modules import create_clip_style_model_global
from downstream.retrieval.run_retrieval import load_image, build_image_transform

vision_model, text_model, tokenizer, model_info = create_clip_style_model_global(
    {"model_name": "smartx_large", "checkpoint": CHECKPOINT}
)
vision_model.to(DEVICE).eval()
text_model.to(DEVICE).eval()

transform = build_image_transform(image_size=IMAGE_SIZE, in_channels=3)

load_report = model_info.get("load_report", {})
print("image embed dim:", model_info["image_embed_dim"])
print("text embed dim :", model_info["text_embed_dim"])
print("missing keys   :", len(load_report.get("missing_keys", [])))
print("skipped (shape):", len(load_report.get("skipped_shape", [])))

### Encoding helpers

Short inline wrappers mirroring `downstream/zero_shot_classification` and `downstream/retrieval`. Both
encoders already L2-normalize their output, so cosine similarity is just a dot product.

In [ ]:
@torch.no_grad()
def encode_texts(texts):
    tokens = tokenizer(list(texts), padding=True, truncation=True, return_tensors="pt")
    tokens = {k: v.to(DEVICE) for k, v in tokens.items()}
    feats = text_model(**tokens).pooler_output
    return F.normalize(feats, dim=-1)

@torch.no_grad()
def encode_images(pil_images):
    batch = torch.stack([transform(img) for img in pil_images]).to(DEVICE)
    feats = vision_model(batch).pooler_output
    return F.normalize(feats, dim=-1)

## 3. Load and preview the demo images

In [ ]:
pairs = [json.loads(line) for line in (ASSETS / "retrieval_pairs.jsonl").read_text().splitlines() if line.strip()]
image_names = [p["image"].split("/")[-1] for p in pairs]
reports = [p["text"] for p in pairs]
pil_images = [load_image(ASSETS / p["image"], in_channels=3) for p in pairs]

fig, axes = plt.subplots(1, len(pil_images), figsize=(4 * len(pil_images), 4))
for ax, img, name in zip(np.atleast_1d(axes), pil_images, image_names):
    ax.imshow(img, cmap="gray")
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Zero-shot classification

For each finding we average the embeddings of its positive prompts and of its negative
prompts, then score an image by `delta = image·positive − image·negative`. A positive
`delta` indicates greater similarity to the positive prompts than to the negative prompts.
Softmax across the candidate findings gives relative similarity scores. These depend on the candidate prompts and are not calibrated disease probabilities.

In [ ]:
spec = json.loads((ASSETS / "zero_shot_labels.json").read_text())
class_names = [l["name"] for l in spec["labels"]]
pos_templates = spec["positive_templates"]  # same templates as downstream/zero_shot_classification
neg_templates = spec["negative_templates"]

def class_weight(templates, class_name):
    prompts = [t.format(class_name=class_name) for t in templates]
    return F.normalize(encode_texts(prompts).mean(dim=0), dim=0)

pos_weights = torch.stack([class_weight(pos_templates, c) for c in class_names])  # [C, D]
neg_weights = torch.stack([class_weight(neg_templates, c) for c in class_names])

img_feats = encode_images(pil_images)                       # [N, D]
delta = (img_feats @ pos_weights.T) - (img_feats @ neg_weights.T)  # [N, C]
scores = torch.softmax(delta, dim=1).cpu().numpy()

In [ ]:
import pandas as pd
df = pd.DataFrame(scores, index=image_names, columns=class_names)
print("Zero-shot relative similarity scores (softmax over candidate findings):")
display(df.style.format("{:.2f}").highlight_max(axis=1, color="#b6e3b6"))

fig, axes = plt.subplots(1, len(image_names), figsize=(4 * len(image_names), 3.2), sharey=True)
for ax, name, row in zip(np.atleast_1d(axes), image_names, scores):
    ax.bar(class_names, row, color="#4c78a8")
    ax.set_ylabel("Relative similarity score")
    ax.set_title(f"{name}\ntop-1: {class_names[int(row.argmax())]}", fontsize=9)
    ax.set_ylim(0, 1); ax.tick_params(axis="x", rotation=30, labelsize=8)
plt.tight_layout(); plt.show()

## 5. Image–text retrieval

Each demo image has a matching report. We embed all images and all reports and build the
cosine-similarity matrix `S[i, j] = image_i · report_j`. A good model puts the largest value
of each row on the diagonal (the image's own report).

In [ ]:
txt_feats = encode_texts(reports)                 # [N, D]
S = (img_feats @ txt_feats.T).cpu().numpy()      # [N, N]

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(S, cmap="viridis")
ax.set_xticks(range(len(reports)), [f"report {i}" for i in range(len(reports))], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(image_names)), image_names, fontsize=8)
ax.set_title("image \u2192 report cosine similarity")
for i in range(S.shape[0]):
    for j in range(S.shape[1]):
        ax.text(j, i, f"{S[i, j]:.2f}", ha="center", va="center",
                color="white" if S[i, j] < S.max() * 0.6 else "black", fontsize=8)
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

In [ ]:
# Ranked reports per image + recall@1 (diagonal = correct match).
ranks = np.argsort(-S, axis=1)
hits = 0
for i, name in enumerate(image_names):
    order = ranks[i]
    correct = order[0] == i
    hits += correct
    ranked = ", ".join(f"report {j}{'*' if j == i else ''}" for j in order)
    print(f"{name:22s} -> {ranked}   {'HIT' if correct else 'miss'}")
print(f"\nimage->text recall@1: {hits}/{len(image_names)} = {hits / len(image_names):.2f}   (* = ground-truth report)")

## 6. Where to go next

- **Full zero-shot evaluation:** `downstream/zero_shot_classification/run_zero_shot_classification.py` with a dataset
  manifest and `downstream/zero_shot_classification/configs/zero_shot_classification.example.yaml`.
- **Full retrieval evaluation:** `downstream/retrieval/run_retrieval.py` (recall@k, MRR,
  median rank over a case manifest).
- **Feature extraction / linear probing:** `downstream/linear_probe_classification/`.

Everything here used the frozen pretrained weights — no task-specific training.